<a href="https://colab.research.google.com/github/hiroaki-com/colab-ollama-private-chat/blob/main/ollama_colab_private_chat_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ollama Colab Private Chat

A notebook for chatting with local LLMs in a secure environment using Google Colab's GPU.

Features:
- ✅ Completely free · No external API dependency · Privacy protected by in-Colab inference
- 🔒 Stateless design — no logs stored (cleared instantly on browser reload)
- 🚀 No setup required — everything from server startup to UI rendering runs within the same notebook

How to use:
1. Select a model in `Model Registry` and run the cell
2. Run `Server` (first-time model download may take a few minutes)
3. Run `Chat UI — Inline` (inside the cell) or `Chat UI — Standalone` (new browser tab) to start chatting

> - `Inline mode`: Runs inside the Colab output cell. Seamlessly switches between fast `Direct` (Colab internal communication) and `Tunnel (Backup)`.
> - `Standalone mode`: Independent UI running in a separate browser tab. A dedicated URL is issued and can be accessed from other devices such as smartphones.

In [ ]:
#@title 📋 Model Registry

# @markdown ### Model Settings
# @markdown Enter model names separated by commas.
# @markdown > Find official model names here: https://ollama.com/search

model_list = "nemotron-3-nano:4b, ministral-3:3b, qwen3.5:4b" #@param {type:"string"}
num_ctx = 4096 #@param {type:"integer"}

AVAILABLE_MODELS = [
    model.strip()
    for model in model_list.split(',')
    if model.strip()
]

if not AVAILABLE_MODELS:
    raise ValueError("❌ Model list is empty. Please enter at least one model.")

import ipywidgets as widgets
from IPython.display import display

header = widgets.HTML(
    '<h3>📦 Select a Model</h3>'
    '<p style="margin: 5px 0 10px 0; font-size: 13px;">'
    'Select one model to launch, then run the next cell.</p>'
)

model_selector = widgets.RadioButtons(
    options=AVAILABLE_MODELS,
    value=AVAILABLE_MODELS[0],
    layout=widgets.Layout(margin='0 0 0 20px')
)

display(widgets.VBox([header, model_selector]))

effective_ctx = num_ctx if num_ctx > 0 else None

print(f"✅ Model list loaded: {len(AVAILABLE_MODELS)} model(s)")
print("➡️ Select a model, then run the next cell (Server).")

In [ ]:
#@title 🚀 Ollama Server

# @markdown Starts Cloudflare Tunnel and issues a public URL for external access (free, no registration or token required).

MAX_HEALTH_RETRIES   = 150
HEALTH_CHECK_TIMEOUT = 2

BLUE  = "\033[34m"
GREEN = "\033[32m"
GRAY  = "\033[90m"
RESET = "\033[0m"

selected_model = model_selector.value

print(f"\nOLLAMA COLAB SERVER 🚀")
print(f"{GRAY}──────────────────────────────────────────{RESET}")

print(f"  {BLUE}◦{RESET} System   Installing dependencies")
!which zstd > /dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq zstd > /dev/null)
!which ollama > /dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!which cloudflared > /dev/null 2>&1 || (wget -qO /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared)

import re, subprocess, time, os, requests, threading

if not re.fullmatch(r'[a-zA-Z0-9._:/-]+', selected_model):
    raise ValueError(f"Invalid model name: {selected_model}")

# Environment variables (CORS and host settings)
os.environ['OLLAMA_HOST']              = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS']           = '*'
os.environ['OLLAMA_KEEP_ALIVE']        = '24h'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_FLASH_ATTENTION']   = '1'
os.environ['OLLAMA_KV_CACHE_TYPE']     = 'q8_0'

!pkill ollama || true
subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=os.environ)

for _ in range(MAX_HEALTH_RETRIES):
    try:
        if requests.get("http://0.0.0.0:11434/api/tags", timeout=HEALTH_CHECK_TIMEOUT).status_code == 200:
            print(f"  {GREEN}✓{RESET} Ready    Ollama server started")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(0.4)
else:
    raise RuntimeError("⚠️ Server health check failed.")

print(f"  {BLUE}◦{RESET} Network  Establishing Cloudflare Tunnel")
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
)
public_url = None
for line in iter(cf_proc.stderr.readline, b''):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line.decode())
    if m:
        public_url = m.group(0)
        break
if not public_url:
    raise RuntimeError("⚠️ Failed to obtain Cloudflare Tunnel URL.")
print(f"  {GREEN}✓{RESET} Public   {public_url}")

print(f"\n  ▶ Model    {selected_model}")
subprocess.run(["/usr/local/bin/ollama", "pull", selected_model], env=os.environ, check=True)

# Warmup and VRAM monitor
print(f"  {BLUE}◦{RESET} VRAM     Loading model into VRAM...", end="", flush=True)
warmup_payload = {"model": selected_model, "prompt": "hi", "stream": False}
if effective_ctx: warmup_payload["options"] = {"num_ctx": effective_ctx}

def check_vram():
    try:
        ps = requests.get("http://0.0.0.0:11434/api/ps").json()
        base = selected_model.split(':')[0]
        return any(m['name'].split(':')[0] == base for m in ps.get('models', []))
    except Exception: return False

MAX_VRAM_WAIT = 90
warmup_thread = threading.Thread(
    target=lambda: requests.post("http://0.0.0.0:11434/api/generate", json=warmup_payload),
    daemon=True
)
warmup_thread.start()

for _ in range(MAX_VRAM_WAIT):
    if not warmup_thread.is_alive() and check_vram():
        print(f"\n  {GREEN}✓{RESET} Loaded   Model loaded into VRAM")
        break
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print(f"\n  ℹ️ Server ready. Model is still loading; you can proceed but the first response may take a moment.")

print(f"\n{GRAY}──────────────────────────────────────────{RESET}")
print(f"STATUS   : {GREEN}RUNNING (Background){RESET}")
print(f"ENDPOINT : {public_url}")
print(f"{GRAY}──────────────────────────────────────────{RESET}")
print("\n✅ Setup complete. Run the next cell.")


In [ ]:
#@title 💬 Chat UI — Inline

# @markdown * `Direct` (recommended): Uses Colab's internal kernel communication for fast operation.
# @markdown * `Tunnel (Backup)` (fallback): Connects via Cloudflare Tunnel. Use this as a backup when Direct communication is unstable.

import base64
import json
import threading
import uuid

import requests
from IPython.display import HTML, display
from google.colab import output

print(f"  {BLUE}◦{RESET} UI       Building Chat UI...")

_model  = model_selector.value
_ctx_js = str(effective_ctx) if effective_ctx else "null"
_max_turns = max(4, ((effective_ctx or 4096) * 3 // 4) // 400 * 2)
_max_turns_js = str(_max_turns)

_stream_jobs = {}
_stream_jobs_lock = threading.Lock()

def _stream_start(model, messages, ctx):
    with _stream_jobs_lock:
        done_keys = [k for k, v in _stream_jobs.items() if v['done']]
        for k in done_keys:
            del _stream_jobs[k]
    job_id = uuid.uuid4().hex
    _stream_jobs[job_id] = {'buf': b'', 'done': False, 'error': None, 'cancel': threading.Event()}
    def _run():
        cancel_ev = _stream_jobs[job_id]['cancel']
        try:
            payload = {"model": model, "messages": messages, "stream": True}
            if ctx:
                payload["options"] = {"num_ctx": int(ctx)}
            with requests.post("http://localhost:11434/api/chat", json=payload, timeout=300, stream=True) as r:
                r.raise_for_status()
                for line in r.iter_lines():
                    if cancel_ev.is_set():
                        break
                    if not line:
                        continue
                    try:
                        chunk = json.loads(line).get('message', {}).get('content', '')
                        if not chunk:
                            continue
                        with _stream_jobs_lock:
                            _stream_jobs[job_id]['buf'] += chunk.encode('utf-8')
                    except Exception:
                        pass
        except Exception as e:
            with _stream_jobs_lock:
                _stream_jobs[job_id]['error'] = str(e)
        finally:
            with _stream_jobs_lock:
                _stream_jobs[job_id]['done'] = True
    threading.Thread(target=_run, daemon=True).start()
    return job_id

def _stream_poll(job_id, offset=0):
    job = _stream_jobs.get(job_id)
    if not job:
        return 'ERR|' + base64.b64encode(b'job not found').decode() + '|'
    with _stream_jobs_lock:
        buf_snapshot = job['buf']
        err_snapshot = job['error'] or ''
        is_done      = job['done']
        if is_done:
            _stream_jobs.pop(job_id, None)
    delta_bytes = buf_snapshot[offset:]
    buf_b64 = base64.b64encode(delta_bytes).decode()
    err_b64 = base64.b64encode(err_snapshot.encode()).decode() if err_snapshot else ''
    result = ('DONE' if is_done else 'WAIT') + '|' + buf_b64 + '|' + err_b64
    return result

output.register_callback("ollama_stream_start", _stream_start)
output.register_callback("ollama_stream_poll",  _stream_poll)

def _stream_cancel(job_id):
    with _stream_jobs_lock:
        job = _stream_jobs.get(job_id)
    if job:
        job['cancel'].set()

output.register_callback("ollama_stream_cancel", _stream_cancel)

_INLINE_UI_HTML = r"""<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans+JP:wght@400;500&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/marked/9.1.6/marked.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/dompurify/3.0.8/purify.min.js"></script>
<style>
:root{--bg:#f5f0e8;--surface:#faf7f2;--border:#c8bfae;--accent:#3a6b4a;--text:#2c2820;--muted:#8a7f72;--user-bg:#e8edf5;--ai-bg:#f0f5ee}
*{box-sizing:border-box;margin:0;padding:0}
#oc{font-family:'IBM Plex Sans JP',sans-serif;background:var(--bg);color:var(--text);max-width:700px;padding:14px}
.oc-ph{font-family:'IBM Plex Mono',monospace;display:flex;justify-content:space-between;align-items:baseline;padding:8px 0 12px;border-bottom:1px solid var(--border);margin-bottom:12px}
.oc-ph .t{font-size:13px;font-weight:600;letter-spacing:.04em}
.oc-ph .s{font-size:11px;color:var(--muted)}
.oc-tabs{display:flex;gap:4px;margin-bottom:10px}
.oc-tab{font-family:'IBM Plex Mono',monospace;font-size:12px;padding:4px 14px;border:1px solid var(--border);background:var(--surface);color:var(--muted);cursor:pointer}
.oc-tab.on{background:var(--accent);color:#fff;border-color:var(--accent)}
.oc-sb{display:flex;align-items:center;gap:8px;margin-bottom:10px;font-size:12px;font-family:'IBM Plex Mono',monospace;color:var(--muted)}
.oc-sb input{flex:1;padding:5px 8px;border:1px solid var(--border);background:var(--surface);color:var(--text);font-family:'IBM Plex Mono',monospace;font-size:12px;outline:none}
.oc-card{border:1px solid var(--border);background:var(--surface)}
.oc-ch{display:flex;justify-content:space-between;align-items:center;padding:8px 12px;border-bottom:1px solid var(--border);font-size:12px;font-family:'IBM Plex Mono',monospace}
.led{display:inline-block;width:8px;height:8px;border-radius:50%;background:#bbb;margin-right:6px;vertical-align:middle}
.led.on{background:var(--accent);animation:p 1.2s ease-in-out infinite}
@keyframes p{0%,100%{opacity:1}50%{opacity:.3}}
.tag{display:inline-block;padding:1px 6px;border:1px solid var(--border);font-size:11px;color:var(--muted);margin-left:4px}
.oc-log{height:340px;overflow-y:auto;padding:12px;display:flex;flex-direction:column;gap:8px}
.sep{text-align:center;font-size:11px;color:var(--muted);font-family:'IBM Plex Mono',monospace;padding:4px 0;border-top:1px dashed var(--border);margin:4px 0}
.bbl{max-width:85%;padding:8px 12px;font-size:13px;line-height:1.65;word-break:break-word}
.bbl.u{align-self:flex-end;background:var(--user-bg);border:1px solid #c5d0e0}
.bbl.a{align-self:flex-start}
.bbl.e{align-self:flex-start;background:var(--ai-bg);border:1px solid #e0b8b8;color:#c0392b}
.bbl pre{background:#e8e4dc;padding:8px;overflow-x:auto;font-family:'IBM Plex Mono',monospace;font-size:12px;margin-top:6px;white-space:pre-wrap}
.tk{display:flex;gap:4px;padding:6px 0;align-items:center;min-height:20px}
.tk div{width:7px;height:7px;background:var(--accent);border-radius:50%;animation:tk 1.4s infinite ease-in-out;opacity:.4}
.tk div:nth-child(2){animation-delay:.2s}
.tk div:nth-child(3){animation-delay:.4s}
@keyframes tk{0%,80%,100%{transform:scale(0.8);opacity:.4}40%{transform:scale(1.2);opacity:1}}
.oc-ia{display:flex;gap:8px;padding:10px 12px;border-top:1px solid var(--border)}
.oc-ia textarea{flex:1;padding:7px 10px;border:1px solid var(--border);background:var(--bg);color:var(--text);font-family:'IBM Plex Sans JP',sans-serif;font-size:13px;resize:none;min-height:38px;max-height:120px;line-height:1.5;outline:none}
.oc-ia textarea::placeholder{font-size:clamp(10px,2.2vw,13px)}
.oc-ia button{padding:7px 18px;background:var(--accent);color:#fff;border:none;font-size:13px;cursor:pointer;font-family:'IBM Plex Mono',monospace;flex-shrink:0}
.oc-ia button:disabled,.oc-ia textarea:disabled{opacity:.5;cursor:not-allowed}
#ocAbortBtn{background:none;border:1px solid var(--border);color:var(--muted)}
#ocAbortBtn:hover{border-color:#c0392b;color:#c0392b}
.oc-cf{display:flex;justify-content:space-between;align-items:center;padding:6px 12px;border-top:1px solid var(--border);font-size:11px;font-family:'IBM Plex Mono',monospace;color:var(--muted)}
.oc-cf button{background:none;border:1px solid var(--border);padding:2px 8px;font-size:11px;color:var(--muted);cursor:pointer}
.oc-note{margin-top:10px;font-size:11px;color:var(--muted);font-family:'IBM Plex Mono',monospace;border:1px dashed var(--border);padding:8px 12px;line-height:1.8}
.bbl p{margin-bottom:8px}.bbl p:last-child{margin-bottom:0}.bbl h1,.bbl h2,.bbl h3{font-weight:600;margin:10px 0 4px;color:inherit}.bbl ul,.bbl ol{margin:6px 0 6px 18px}.bbl li{margin-bottom:2px}.bbl code{background:#e8e4dc;padding:1px 4px;font-family:'IBM Plex Mono',monospace;font-size:12px}.bbl blockquote{border-left:3px solid var(--border);padding-left:8px;color:var(--muted);margin:6px 0}.sep{cursor:pointer}.sep:hover{color:var(--accent)}.u-turn{display:flex;flex-direction:column;align-self:flex-end;gap:2px}.u-wrap{display:flex;align-items:flex-start;gap:6px}.u-foot{display:flex;justify-content:flex-end}.cp-btn{display:inline-block;margin:0;padding:3px 7px;font-size:13px;background:none;border:1px solid transparent;border-radius:6px;color:var(--muted);cursor:pointer;font-family:'IBM Plex Mono',monospace;opacity:0;transition:opacity .15s}.cp-btn:hover{background:rgba(0,0,0,.06);border-color:rgba(0,0,0,.15);color:var(--muted)}.bbl:hover .cp-btn{opacity:1}.u-turn:hover .cp-btn{opacity:1}
.edit-ta{width:100%;min-height:60px;max-height:200px;padding:6px 8px;font-family:'IBM Plex Sans JP',sans-serif;font-size:13px;background:var(--bg);color:var(--text);border:1px solid var(--accent);resize:vertical;outline:none;box-sizing:border-box}
</style>
<div id="oc">
  <div class="oc-ph">
    <span class="t">OLLAMA COLAB PRIVATE CHAT</span>
    <span class="s">no history · local inference</span>
  </div>
  <div class="oc-tabs">
    <button id="tabInline" class="oc-tab on" onclick="ocMode('inline')">[ Direct ]</button>
    <button id="tabTunnel" class="oc-tab" onclick="ocMode('tunnel')">[ Tunnel (Backup) ]</button>
  </div>
  <div class="oc-sb">
    BASE URL
    <input id="ocUrl" type="text" readonly>
  </div>
  <div class="oc-card">
    <div class="oc-ch">
      <span><span id="ocLed" class="led"></span></span>
      <span><span class="tag" id="ocMTag"></span></span>
    </div>
    <div id="ocLog" class="oc-log"></div>
    <div class="oc-ia">
      <textarea id="ocIn" rows="1" placeholder="Ask me anything"></textarea>
      <button id="ocBtn" onclick="ocSend()">Send</button>
      <button id="ocAbortBtn" onclick="ocAbort()" style="display:none">Stop</button>
    </div>
    <div class="oc-cf">
      <span>stateless · no localStorage</span>
      <button onclick="ocClear()">Clear</button>
    </div>
  </div>
  <div id="ocNt" class="oc-note">
    ℹ️ Direct is the default mode and uses Colab's internal communication. Select Tunnel (Backup) if Direct is unstable.
  </div>
</div>
<script>
(function(){
const MODEL="__MODEL__", CTX=__CTX_JS__, MAX_TURNS=__MAX_TURNS__, TUNNEL_URL="__TUNNEL_URL__";
let currentMode = 'inline';
const $=id=>document.getElementById(id);
const elLog=$('ocLog'), elIn=$('ocIn'), elBtn=$('ocBtn'), elLed=$('ocLed'), elNt=$('ocNt'), elUrl=$('ocUrl');
const elAbortBtn = $('ocAbortBtn');
let _abort = null;
let _history = [];
window.ocAbort = function() { if (_abort) _abort(); };

$('ocMTag').textContent = MODEL;
elUrl.value = '(Direct: Python kernel)';

window.ocMode = function(m) {
  currentMode = m;
  $('tabInline').classList.toggle('on', m==='inline');
  $('tabTunnel').classList.toggle('on', m==='tunnel');
  elUrl.value = m === 'inline' ? '(Direct: Python kernel)' : TUNNEL_URL;
  if (m === 'inline') {
    elNt.innerHTML = 'ℹ️ Direct is the default mode and uses Colab's internal communication. Select Tunnel (Backup) if Direct is unstable.';
  } else {
    elNt.innerHTML = '⚠️ Tunnel (Backup) mode: If you re-run Cell 2, re-run Cell 3 as well (the URL changes).';
  }
  elNt.style.display = '';
};

window.ocClear = function() { elLog.innerHTML = ''; _history = []; };
function ledOn(v) { elLed.className = 'led' + (v ? ' on' : ''); }
function md(s) { return DOMPurify.sanitize(marked.parse(s)); }
function addSep() {
  const d = document.createElement('div');
  d.className = 'sep';
  d.textContent = '↻ Reset';
  d.onclick = function(){
    while (elLog.firstChild && elLog.firstChild !== d)
      elLog.removeChild(elLog.firstChild);
    d.remove();
  };
  elLog.appendChild(d);
}
function addBubble(cls, content, asHtml) {
  const d = document.createElement('div'); d.className = 'bbl ' + cls;
  if (asHtml) d.innerHTML = content; else d.textContent = content;
  elLog.appendChild(d); return d;
}
function setDisabled(v) {
  elBtn.disabled = v;
  elAbortBtn.style.display = v ? '' : 'none';
}

window.ocSend = async function() {
  const t = elIn.value.trim();
  if (!t) return;
  elIn.value = ''; elIn.style.height = '';
  const historyLenBefore = _history.length;
  _history.push({ role: 'user', content: t });
  if (elLog.children.length) addSep();
  const turnDiv = document.createElement('div');
  turnDiv.className = 'u-turn';
  const retryBtn = document.createElement('button');
  retryBtn.className = 'cp-btn'; retryBtn.textContent = '↻'; retryBtn.title = 'Retry';
  function doRetry(newText) {
    if (elBtn.disabled) return;
    const prev = turnDiv.previousSibling;
    if (prev && prev.classList.contains('sep')) prev.remove();
    while (turnDiv.nextSibling) elLog.removeChild(turnDiv.nextSibling);
    _history.length = historyLenBefore;
    turnDiv.remove();
    elIn.value = newText;
    ocSend();
  }
  retryBtn.onclick = function() { doRetry(t); };
  const userBubble = document.createElement('div');
  userBubble.className = 'bbl u';
  userBubble.textContent = t;
  const wrapDiv = document.createElement('div');
  wrapDiv.className = 'u-wrap';
  wrapDiv.appendChild(retryBtn);
  wrapDiv.appendChild(userBubble);
  const foot = document.createElement('div');
  foot.className = 'u-foot';
  const copyUserBtn = document.createElement('button');
  copyUserBtn.className = 'cp-btn'; copyUserBtn.textContent = '⧉'; copyUserBtn.title = 'Copy';
  copyUserBtn.onclick = function() { navigator.clipboard.writeText(t).then(function(){ copyUserBtn.textContent = '✓'; setTimeout(function(){ copyUserBtn.textContent = '⧉'; }, 1500); }); };
  foot.appendChild(copyUserBtn);

  const editBtn = document.createElement('button');
  editBtn.className = 'cp-btn'; editBtn.textContent = '✎'; editBtn.title = 'Edit';
  editBtn.onclick = enterEditMode;
  foot.appendChild(editBtn);

  function enterEditMode() {
    const ta = document.createElement('textarea');
    ta.className = 'edit-ta';
    ta.value = t;
    userBubble.replaceWith(ta);
    ta.focus(); ta.select();

    copyUserBtn.style.display = 'none';
    editBtn.style.display = 'none';

    const confirmBtn = document.createElement('button');
    confirmBtn.className = 'cp-btn'; confirmBtn.textContent = '✓'; confirmBtn.title = 'Confirm';
    const cancelBtn = document.createElement('button');
    cancelBtn.className = 'cp-btn'; cancelBtn.textContent = '✕'; cancelBtn.title = 'Cancel';

    function exitEditMode() {
      ta.replaceWith(userBubble);
      copyUserBtn.style.display = '';
      editBtn.style.display = '';
      confirmBtn.remove();
      cancelBtn.remove();
    }

    confirmBtn.onclick = function() {
      const newText = ta.value.trim();
      if (newText) doRetry(newText);
      else exitEditMode();
    };
    cancelBtn.onclick = exitEditMode;

    ta.addEventListener('keydown', function(e) {
      if (e.key === 'Enter' && (e.ctrlKey || e.metaKey)) { e.preventDefault(); confirmBtn.onclick(); }
    });

    foot.appendChild(confirmBtn);
    foot.appendChild(cancelBtn);
  }
  turnDiv.appendChild(wrapDiv);
  turnDiv.appendChild(foot);
  elLog.appendChild(turnDiv);
  elLog.scrollTop = elLog.scrollHeight;

  const ai = addBubble('a', '<div class="tk"><div></div><div></div><div></div></div>', true);
  elLog.scrollTop = elLog.scrollHeight;
  setDisabled(true); ledOn(true);

  let full = '';
  try {
    let _rafId = null;
    if (currentMode === 'inline') {
      // Direct mode: streaming via Python kernel (Base64 polling)
      const startRaw = await google.colab.kernel.invokeFunction('ollama_stream_start', [MODEL, _history.slice(-MAX_TURNS), CTX], {});
      const jobId = startRaw.data['text/plain'].slice(1, -1);
      let firstChunk = true;
      let _abortA = false, offset = 0;
      _abort = () => { _abortA = true; google.colab.kernel.invokeFunction('ollama_stream_cancel', [jobId], {}); };
      while (true) {
        if (_abortA) break;
        const pollRaw = await google.colab.kernel.invokeFunction('ollama_stream_poll', [jobId, offset], {});
        const s = pollRaw.data['text/plain'].slice(1, -1);
        const pipe1 = s.indexOf('|'), pipe2 = s.indexOf('|', pipe1 + 1);
        const status = s.slice(0, pipe1);
        const deltaB64 = s.slice(pipe1 + 1, pipe2);
        const errB64 = s.slice(pipe2 + 1);
        if (deltaB64) {
          const deltaBytes = Uint8Array.from(atob(deltaB64), c => c.charCodeAt(0));
          const delta = new TextDecoder().decode(deltaBytes);
          if (delta) {
            full += delta;
            offset += deltaBytes.length;
            if (firstChunk) { ai.innerHTML = ''; firstChunk = false; }
            ai.innerHTML = md(full);
          }
        }
        if (status === 'ERR') {
          const errMsg = errB64 ? new TextDecoder().decode(Uint8Array.from(atob(errB64), c => c.charCodeAt(0))) : 'stream job not found';
          throw new Error(errMsg);
        }
        if (status === 'DONE') {
          if (errB64) {
            const errMsg = new TextDecoder().decode(Uint8Array.from(atob(errB64), c => c.charCodeAt(0)));
            if (errMsg) throw new Error(errMsg);
          }
          break;
        }
        if (!deltaB64) await new Promise(r => setTimeout(r, 150));
      }
      if (_abortA) _history.length = historyLenBefore;
    } else {
      // Tunnel mode (Backup): stream directly from Cloudflare Tunnel URL via fetch
      const u = elUrl.value.replace(/\/$/, '');
      const body = { model: MODEL, messages: _history.slice(-MAX_TURNS), stream: true };
      if (CTX) body.options = { num_ctx: CTX };
      const ctrl = new AbortController();
      _abort = () => ctrl.abort();
      const res = await fetch(u + '/api/chat', {
        method: 'POST',
        headers: {
          'Content-Type': 'application/json'
        },
        body: JSON.stringify(body),
        signal: ctrl.signal
      });
      if (!res.ok) throw new Error('HTTP ' + res.status);
      const ct = res.headers.get('content-type') || '';
      if (!ct.includes('application/json') && !ct.includes('application/x-ndjson') && !ct.includes('text/plain')) {
        throw new Error('Unexpected Content-Type "' + ct + '" — the tunnel interstitial page may have been returned.');
      }
      const reader = res.body.getReader(), dec = new TextDecoder();
      let buf = '', firstToken = true;
      while (true) {
        const { done, value } = await reader.read();
        if (done) break;
        buf += dec.decode(value, { stream: true });
        const lines = buf.split('\n');
        buf = lines.pop();
        for (const l of lines) {
          if (!l.trim()) continue;
          try {
            const j = JSON.parse(l);
            const chunk = j.message?.content || '';
            if (!chunk) {
              if (j.done) break;
              continue;
            }
            if (firstToken) { ai.innerHTML = ''; firstToken = false; }
            full += chunk;
            if (_rafId === null) {
              _rafId = requestAnimationFrame(() => { ai.innerHTML = md(full); _rafId = null; });
            }
            if (j.done) break;
          } catch(_) {}
        }
      }
    }
    if (full) {
      if (_rafId !== null) { cancelAnimationFrame(_rafId); _rafId = null; }
      ai.innerHTML = md(full);
      _history.push({ role: 'assistant', content: full });
      const btn = document.createElement('button');
      btn.className = 'cp-btn'; btn.textContent = '⧉'; btn.title = 'Copy';
      btn.onclick = function() { navigator.clipboard.writeText(full).then(function(){ btn.textContent = '✓'; setTimeout(function(){ btn.textContent = '⧉'; }, 1500); }); };
      ai.appendChild(btn);
    } else { ai.textContent = '(empty response)'; }
  } catch(e) {
    _history.length = historyLenBefore;
    if (e.name === 'AbortError') {
      if (full) { ai.innerHTML = md(full); }
      else { ai.innerHTML = ''; ai.textContent = '(stopped)'; }
    } else {
      ai.className = 'bbl e';
      ai.textContent = 'Error: ' + e.message;
    }
  } finally {
    _abort = null;
    setDisabled(false); ledOn(false);
  }
};
elIn.addEventListener('keydown', e => {
  if (e.key === 'Enter' && (e.ctrlKey || e.metaKey)) { e.preventDefault(); ocSend(); return; }
  if (e.key === 'Enter' && !e.shiftKey && !e.isComposing) { e.preventDefault(); ocSend(); }
});
elIn.addEventListener('input', function() { this.style.height = ''; this.style.height = Math.min(this.scrollHeight, 120) + 'px'; });
elIn.focus();
document.addEventListener('touchstart', () => elIn.focus(), { once: true });
})();
</script>"""

_html = _INLINE_UI_HTML \
    .replace("__MODEL__",     _model) \
    .replace("__CTX_JS__",    _ctx_js) \
    .replace("__MAX_TURNS__", _max_turns_js) \
    .replace("__TUNNEL_URL__",     public_url)

print(f"  {GREEN}✓{RESET} Ready    Chat UI ready")
display(HTML(_html))

In [ ]:
#@title 🌐 Chat UI — Standalone

# @markdown - A standalone UI that uses a full browser tab for chatting.
# @markdown - Access it via the issued Cloudflare Tunnel URL (can also be shared with other devices).

print(f"  {BLUE}◦{RESET} System   Checking dependencies")
!pip install -q fastapi uvicorn httpx
print(f"  {GREEN}✓{RESET} System   Dependencies ready")

!pkill -f "uvicorn.*11435"     > /dev/null 2>&1 || true
!pkill -f "cloudflared.*11435" > /dev/null 2>&1 || true

import re
import subprocess
import threading
import time

import httpx
import requests
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse, StreamingResponse
from IPython.display import HTML, display

time.sleep(1)

_model  = model_selector.value
_ctx_js = str(effective_ctx) if effective_ctx else "null"
_max_turns = max(4, ((effective_ctx or 4096) * 3 // 4) // 400 * 2)
_max_turns_js = str(_max_turns)

_STANDALONE_UI_HTML = r"""<!DOCTYPE html>
<html lang='en'><head>
<meta charset='UTF-8'>
<meta name='viewport' content='width=device-width, initial-scale=1, maximum-scale=1, user-scalable=no'>
<title>Ollama Colab Private Chat — Standalone</title>
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans+JP:wght@400;500&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/marked/9.1.6/marked.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/dompurify/3.0.8/purify.min.js"></script>
<style>
:root{--bg:#f5f0e8;--surface:#faf7f2;--border:#c8bfae;--accent:#3a6b4a;--text:#2c2820;--muted:#8a7f72;--user-bg:#e8edf5;--ai-bg:#f0f5ee}
*{box-sizing:border-box;margin:0;padding:0}
body{margin:0;background:var(--bg)}
#oc{font-family:'IBM Plex Sans JP',sans-serif;color:var(--text);max-width:1200px;width:100%;margin:0 auto;padding:clamp(12px,2vw,24px);min-height:100vh;display:flex;flex-direction:column}
.oc-ph{font-family:'IBM Plex Mono',monospace;display:flex;justify-content:space-between;align-items:baseline;padding:8px 0 12px;border-bottom:1px solid var(--border);margin-bottom:12px}
.oc-ph .t{font-size:13px;font-weight:600;letter-spacing:.04em}
.oc-ph .s{font-size:11px;color:var(--muted)}
.oc-sb{display:flex;align-items:center;gap:8px;margin-bottom:10px;font-size:12px;font-family:'IBM Plex Mono',monospace;color:var(--muted)}
.oc-sb input{flex:1;padding:5px 8px;border:1px solid var(--border);background:var(--surface);color:var(--text);font-family:'IBM Plex Mono',monospace;font-size:12px;outline:none}
.oc-card{border:1px solid var(--border);background:var(--surface);flex:1;display:flex;flex-direction:column}
.oc-ch{display:flex;justify-content:space-between;align-items:center;padding:8px 12px;border-bottom:1px solid var(--border);font-size:12px;font-family:'IBM Plex Mono',monospace}
.led{display:inline-block;width:8px;height:8px;border-radius:50%;background:#bbb;margin-right:6px;vertical-align:middle}
.led.on{background:var(--accent);animation:p 1.2s ease-in-out infinite}
@keyframes p{0%,100%{opacity:1}50%{opacity:.3}}
.tag{display:inline-block;padding:1px 6px;border:1px solid var(--border);font-size:11px;color:var(--muted);margin-left:4px}
.oc-log{flex:1;min-height:240px;overflow-y:auto;padding:12px;display:flex;flex-direction:column;gap:8px}
.sep{text-align:center;font-size:11px;color:var(--muted);font-family:'IBM Plex Mono',monospace;padding:4px 0;border-top:1px dashed var(--border);margin:4px 0}
.bbl{max-width:85%;padding:8px 12px;font-size:13px;line-height:1.65;word-break:break-word}
.bbl.u{align-self:flex-end;background:var(--user-bg);border:1px solid #c5d0e0}
.bbl.a{align-self:flex-start}
.bbl.e{align-self:flex-start;background:var(--ai-bg);border:1px solid #e0b8b8;color:#c0392b}
.bbl pre{background:#e8e4dc;padding:8px;overflow-x:auto;font-family:'IBM Plex Mono',monospace;font-size:12px;margin-top:6px;white-space:pre-wrap}
.tk{display:flex;gap:4px;padding:6px 0;align-items:center;min-height:20px}
.tk div{width:7px;height:7px;background:var(--accent);border-radius:50%;animation:tk 1.4s infinite ease-in-out;opacity:.4}
.tk div:nth-child(2){animation-delay:.2s}
.tk div:nth-child(3){animation-delay:.4s}
@keyframes tk{0%,80%,100%{transform:scale(0.8);opacity:.4}40%{transform:scale(1.2);opacity:1}}
.oc-ia{display:flex;gap:8px;padding:10px 12px;border-top:1px solid var(--border)}
.oc-ia textarea{flex:1;padding:7px 10px;border:1px solid var(--border);background:var(--bg);color:var(--text);font-family:'IBM Plex Sans JP',sans-serif;font-size:13px;resize:none;min-height:38px;max-height:120px;line-height:1.5;outline:none}
.oc-ia textarea::placeholder{font-size:clamp(10px,2.2vw,13px)}
.oc-ia button{padding:7px 18px;background:var(--accent);color:#fff;border:none;font-size:13px;cursor:pointer;font-family:'IBM Plex Mono',monospace;flex-shrink:0}
.oc-ia button:disabled,.oc-ia textarea:disabled,.disabled-btn{opacity:.5;pointer-events:none}
#ocAbortBtn{background:none;border:1px solid var(--border);color:var(--muted)}
#ocAbortBtn:hover{border-color:#c0392b;color:#c0392b}
.oc-cf{display:flex;justify-content:space-between;align-items:center;padding:6px 12px;border-top:1px solid var(--border);font-size:11px;font-family:'IBM Plex Mono',monospace;color:var(--muted)}
.oc-cf button{background:none;border:1px solid var(--border);padding:2px 8px;font-size:11px;color:var(--muted);cursor:pointer}
.bbl p{margin-bottom:8px}.bbl p:last-child{margin-bottom:0}.bbl h1,.bbl h2,.bbl h3{font-weight:600;margin:10px 0 4px}.bbl ul,.bbl ol{margin:6px 0 6px 18px}.bbl li{margin-bottom:2px}.bbl code{background:#e8e4dc;padding:1px 4px;font-family:'IBM Plex Mono',monospace;font-size:12px}.bbl blockquote{border-left:3px solid var(--border);padding-left:8px;color:var(--muted);margin:6px 0}.sep{cursor:pointer}.sep:hover{color:var(--accent)}.u-turn{display:flex;flex-direction:column;align-self:flex-end;gap:2px}.u-wrap{display:flex;align-items:flex-start;gap:6px}.u-foot{display:flex;justify-content:flex-end}.cp-btn{display:inline-block;margin:0;padding:3px 7px;font-size:13px;background:none;border:1px solid transparent;border-radius:6px;color:var(--muted);cursor:pointer;font-family:'IBM Plex Mono',monospace}.cp-btn:hover{background:rgba(0,0,0,.06);border-color:rgba(0,0,0,.15);color:var(--muted)}
.edit-ta{width:100%;min-height:60px;max-height:200px;padding:6px 8px;font-family:'IBM Plex Sans JP',sans-serif;font-size:13px;background:var(--bg);color:var(--text);border:1px solid var(--accent);resize:vertical;outline:none;box-sizing:border-box}
@media (max-width:600px){.oc-sb{display:none}}
</style>
</head>
<body>
<div id="oc">
  <div class="oc-ph">
    <span class="t">OLLAMA COLAB PRIVATE CHAT</span>
    <span class="s">no history · local inference</span>
  </div>
  <div class="oc-sb">
    BASE URL
    <input id="ocUrl" type="text" value="" readonly>
  </div>
  <div class="oc-card">
    <div class="oc-ch">
      <span><span id="ocLed" class="led"></span></span>
      <span><span class="tag" id="ocMTag"></span></span>
    </div>
    <div id="ocLog" class="oc-log"></div>
    <div class="oc-ia">
      <textarea id="ocIn" rows="1" placeholder="Ask me anything" autofocus></textarea>
      <button id="ocBtn" onclick="ocSend()">Send</button>
      <button id="ocAbortBtn" onclick="ocAbort()" style="display:none">Stop</button>
    </div>
    <div class="oc-cf">
      <span>stateless · no localStorage</span>
      <button onclick="ocClear()">Clear</button>
    </div>
  </div>
</div>
<script>
(function(){
const MODEL="__MODEL__", CTX=__CTX_JS__, MAX_TURNS=__MAX_TURNS__;
const $=id=>document.getElementById(id);
const elLog=$('ocLog'), elIn=$('ocIn'), elBtn=$('ocBtn'), elLed=$('ocLed'), elUrl=$('ocUrl');
const elAbortBtn = $('ocAbortBtn');
let _abort = null;
let _history = [];
window.ocAbort = function() { if (_abort) _abort(); };

elUrl.value = window.location.origin;
$('ocMTag').textContent = MODEL;

window.ocClear = function() { elLog.innerHTML = ''; _history = []; };
function ledOn(v) { elLed.className = 'led' + (v ? ' on' : ''); }
function md(s) { return DOMPurify.sanitize(marked.parse(s)); }
function addSep() {
  const d = document.createElement('div');
  d.className = 'sep';
  d.textContent = '↻ Reset';
  d.onclick = function(){
    while (elLog.firstChild && elLog.firstChild !== d)
      elLog.removeChild(elLog.firstChild);
    d.remove();
  };
  elLog.appendChild(d);
}
function addBubble(cls, content, asHtml) {
  const d = document.createElement('div'); d.className = 'bbl ' + cls;
  if (asHtml) d.innerHTML = content; else d.textContent = content;
  elLog.appendChild(d); return d;
}
function setDisabled(v) {
  v ? elBtn.classList.add('disabled-btn') : elBtn.classList.remove('disabled-btn');
  elAbortBtn.style.display = v ? '' : 'none';
}

window.ocSend = async function() {
  if (_abort) return;
  const t = elIn.value.trim();
  if (!t) return;
  elIn.value = ''; elIn.style.height = ''; elBtn.focus();
  const historyLenBefore = _history.length;
  _history.push({ role: 'user', content: t });
  if (elLog.children.length) addSep();
  const turnDiv = document.createElement('div');
  turnDiv.className = 'u-turn';
  const retryBtn = document.createElement('button');
  retryBtn.className = 'cp-btn'; retryBtn.textContent = '↻'; retryBtn.title = 'Retry';
  function doRetry(newText) {
    if (_abort) return;
    const prev = turnDiv.previousSibling;
    if (prev && prev.classList.contains('sep')) prev.remove();
    while (turnDiv.nextSibling) elLog.removeChild(turnDiv.nextSibling);
    _history.length = historyLenBefore;
    turnDiv.remove();
    elIn.value = newText;
    ocSend();
  }
  retryBtn.onclick = function() { doRetry(t); };
  const userBubble = document.createElement('div');
  userBubble.className = 'bbl u';
  userBubble.textContent = t;
  const wrapDiv = document.createElement('div');
  wrapDiv.className = 'u-wrap';
  wrapDiv.appendChild(retryBtn);
  wrapDiv.appendChild(userBubble);
  const foot = document.createElement('div');
  foot.className = 'u-foot';
  const copyUserBtn = document.createElement('button');
  copyUserBtn.className = 'cp-btn'; copyUserBtn.textContent = '⧉'; copyUserBtn.title = 'Copy';
  copyUserBtn.onclick = function() { navigator.clipboard.writeText(t).then(function(){ copyUserBtn.textContent = '✓'; setTimeout(function(){ copyUserBtn.textContent = '⧉'; }, 1500); }); };
  foot.appendChild(copyUserBtn);

  const editBtn = document.createElement('button');
  editBtn.className = 'cp-btn'; editBtn.textContent = '✎'; editBtn.title = 'Edit';
  editBtn.onclick = enterEditMode;
  foot.appendChild(editBtn);

  function enterEditMode() {
    const ta = document.createElement('textarea');
    ta.className = 'edit-ta';
    ta.value = t;
    userBubble.replaceWith(ta);
    ta.focus(); ta.select();

    copyUserBtn.style.display = 'none';
    editBtn.style.display = 'none';

    const confirmBtn = document.createElement('button');
    confirmBtn.className = 'cp-btn'; confirmBtn.textContent = '✓'; confirmBtn.title = 'Confirm';
    const cancelBtn = document.createElement('button');
    cancelBtn.className = 'cp-btn'; cancelBtn.textContent = '✕'; cancelBtn.title = 'Cancel';

    function exitEditMode() {
      ta.replaceWith(userBubble);
      copyUserBtn.style.display = '';
      editBtn.style.display = '';
      confirmBtn.remove();
      cancelBtn.remove();
    }

    confirmBtn.onclick = function() {
      const newText = ta.value.trim();
      if (newText) doRetry(newText);
      else exitEditMode();
    };
    cancelBtn.onclick = exitEditMode;

    ta.addEventListener('keydown', function(e) {
      if (e.key === 'Enter' && (e.ctrlKey || e.metaKey)) { e.preventDefault(); confirmBtn.onclick(); }
    });

    foot.appendChild(confirmBtn);
    foot.appendChild(cancelBtn);
  }
  turnDiv.appendChild(wrapDiv);
  turnDiv.appendChild(foot);
  elLog.appendChild(turnDiv);
  elLog.scrollTop = elLog.scrollHeight;

  const ai = addBubble('a', '<div class="tk"><div></div><div></div><div></div></div>', true);
  setTimeout(function(){ ai.scrollIntoView({ block: 'start', behavior: 'instant' }); }, 0);
  setDisabled(true); ledOn(true);

  let full = '';
  try {
    const u = elUrl.value.replace(/[/]$/, '');
    const body = { model: MODEL, messages: _history.slice(-MAX_TURNS), stream: true };
    if (CTX) body.options = { num_ctx: CTX };
    const ctrl = new AbortController();
    _abort = () => ctrl.abort();
    const res = await fetch(u + '/api/chat', {
      method: 'POST',
      headers: {
        'Content-Type': 'application/json'
      },
      body: JSON.stringify(body),
      signal: ctrl.signal
    });
    if (!res.ok) throw new Error('HTTP ' + res.status);
    const reader = res.body.getReader(), dec = new TextDecoder();
    let buf = '', firstToken = true;
    let _rafId = null;
    while (true) {
      const { done, value } = await reader.read();
      if (done) break;
      buf += dec.decode(value, { stream: true });
      const lines = buf.split('\n');
      buf = lines.pop();
      for (const l of lines) {
        if (!l.trim()) continue;
        try {
          const j = JSON.parse(l);
          const chunk = j.message?.content || '';
          if (!chunk) continue;
          if (firstToken) { ai.innerHTML = ''; firstToken = false; }
          full += chunk;
          if (_rafId === null) {
            _rafId = requestAnimationFrame(() => { ai.innerHTML = md(full); _rafId = null; });
          }
        } catch(_) {}
      }
    }
    if (full) {
      if (_rafId !== null) { cancelAnimationFrame(_rafId); _rafId = null; }
      ai.innerHTML = md(full);
      _history.push({ role: 'assistant', content: full });
      const btn = document.createElement('button');
      btn.className = 'cp-btn'; btn.textContent = '⧉'; btn.title = 'Copy';
      btn.onclick = function() { navigator.clipboard.writeText(full).then(function(){ btn.textContent = '✓'; setTimeout(function(){ btn.textContent = '⧉'; }, 1500); }); };
      ai.appendChild(btn);
    } else { ai.textContent = '(empty response)'; }
  } catch(e) {
    _history.length = historyLenBefore;
    if (e.name === 'AbortError') {
      if (full) { ai.innerHTML = md(full); }
      else { ai.innerHTML = ''; ai.textContent = '(stopped)'; }
    } else {
      ai.className = 'bbl e';
      ai.textContent = 'Error: ' + e.message;
    }
  } finally {
    _abort = null;
    setDisabled(false); ledOn(false);
  }
};
elIn.addEventListener('keydown', e => {
  if (e.key === 'Enter' && (e.ctrlKey || e.metaKey)) { e.preventDefault(); ocSend(); return; }
  if (e.key === 'Enter' && !e.shiftKey && !e.isComposing) { e.preventDefault(); ocSend(); }
});
elIn.addEventListener('input', function() { this.style.height = ''; this.style.height = Math.min(this.scrollHeight, 120) + 'px'; });
elIn.focus();

// Double-tap outside the form to focus the textarea
let _lastClick = 0;
document.getElementById('oc').addEventListener('click', function(e) {
  let now = Date.now();
  if (now - _lastClick < 300 && !_abort && !e.target.closest('.oc-ia, .cp-btn, .edit-ta, button')) {
    elIn.focus();
  }
  _lastClick = now;
});
})();
</script>
</body></html>"""

# ── FastAPI app ──────────────────────────────────────────────────
_OLLAMA_B = "http://localhost:11434"
_share_app = FastAPI()

@_share_app.get("/")
async def _index_b():
    return HTMLResponse(_html_b)

@_share_app.post("/api/{path:path}")
async def _proxy_b(path: str, request: Request):
    body = await request.body()
    async def _stream():
        async with httpx.AsyncClient(timeout=300) as _c:
            async with _c.stream(
                "POST", f"{_OLLAMA_B}/api/{path}",
                content=body,
                headers={"Content-Type": "application/json"},
            ) as _r:
                async for _chunk in _r.aiter_bytes():
                    yield _chunk
    return StreamingResponse(_stream(), media_type="application/x-ndjson")

# ── Generate HTML (no __TUNNEL_URL__ substitution needed) ─────────
_html_b = (
    _STANDALONE_UI_HTML
    .replace("__MODEL__",     _model)
    .replace("__CTX_JS__",    _ctx_js)
    .replace("__MAX_TURNS__", _max_turns_js)
)

# ── Start uvicorn ────────────────────────────────────────────────
print(f"  {BLUE}◦{RESET} Server   Starting FastAPI (port 11435)")
threading.Thread(
    target=lambda: uvicorn.run(_share_app, host="0.0.0.0", port=11435, log_level="error"),
    daemon=True,
).start()

for _ in range(30):
    try:
        if requests.get("http://localhost:11435/", timeout=1).status_code == 200:
            print(f"  {GREEN}✓{RESET} Server   FastAPI started")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(0.3)
else:
    raise RuntimeError("⚠️ FastAPI startup check failed.")

# ── Cloudflare Tunnel (port 11435) ───────────────────────────────
print(f"  {BLUE}◦{RESET} Network  Establishing Cloudflare Tunnel (port 11435)")
_cf_b = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11435"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE,
)
_url_b = None
for _line in iter(_cf_b.stderr.readline, b""):
    _m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", _line.decode())
    if _m:
        _url_b = _m.group(0)
        break
if not _url_b:
    raise RuntimeError("⚠️ Failed to obtain Cloudflare Tunnel URL.")

# ── DNS reachability check ────────────────────────────────────────
_DNS_RETRIES  = 20
_DNS_INTERVAL = 3
print(f"  {BLUE}◦{RESET} Network  Checking DNS propagation", end="", flush=True)
for _ in range(_DNS_RETRIES):
    try:
        _r = requests.get(_url_b, timeout=4, allow_redirects=False)
        if _r.status_code < 500:
            print(f"\n  {GREEN}✓{RESET} Network  DNS resolved and reachable")
            break
    except requests.exceptions.RequestException:
        pass
    print(".", end="", flush=True)
    time.sleep(_DNS_INTERVAL)
else:
    print(f"\n  ⚠️ DNS check timed out. The URL will be shown, but access may take a few seconds.")
print(f"  {GREEN}✓{RESET} Network  Tunnel established")

# ── Output ──────────────────────────────────────────────────────
print(f"\n{GRAY}──────────────────────────────────────────{RESET}")
print(f"Standalone URL : {GREEN}{_url_b}{RESET}")
print(f"{GRAY}──────────────────────────────────────────{RESET}")

display(HTML(
    f'<div style="display:inline-flex;flex-direction:column;border:1px solid #c8bfae;'
    f'background:#faf7f2;color:#2c2820;font-family:monospace;font-size:11px;margin-top:4px">'
    f'<div style="padding:6px 12px;border-bottom:1px solid #c8bfae;display:flex;align-items:center;gap:8px">'
    f'<span style="width:7px;height:7px;border-radius:50%;background:#3a6b4a;display:inline-block;flex-shrink:0"></span>'
    f'<span style="letter-spacing:.04em">&#x2713; Chat UI &mdash; Standalone</span></div>'
    f'<a href="{_url_b}" target="_blank" style="text-decoration:none;color:#2c2820;'
    f'display:flex;align-items:center;justify-content:space-between;padding:8px 12px;gap:24px">'
    f'<span style="overflow:hidden;text-overflow:ellipsis;white-space:nowrap">{_url_b}</span>'
    f'<span style="flex-shrink:0">&#x2197;</span></a></div>'
))
